In [2]:
# Script to generate n-ary masks per cell type AND save XML metadata to docs and JSON Files used for later analysis
# Process whole slide images

import os
import csv
import json
from datetime import datetime
import numpy as np
import openslide
from glob import glob
import cv2
from shapely.geometry import Polygon
from skimage import draw
import xml.etree.ElementTree as ET

# configuration
data_path = '../EC500/EC500 AI guided whole slide imaging analysis/TrainingData'  # Path to read data from (root containing patient folders)
destination_path = '../EC500/EC500 AI guided whole slide imaging analysis/Output_Folder'  # Path to save masks and metadata
save_tif_slide = False                   # Whether to save slide as TIFF
mask_dtype = np.uint16                  # 16-bit mask to hold many instances
# ---------------------------------

# Create output root and masks folder
os.makedirs(destination_path, exist_ok=True)
masks_root = os.path.join(destination_path, 'masks')
os.makedirs(masks_root, exist_ok=True)

# create metadata root
meta_root = os.path.join(destination_path, 'metadata')
os.makedirs(meta_root, exist_ok=True)

# master CSV path
master_csv_path = os.path.join(meta_root, 'slides_metadata_master.csv')
master_csv_header = [
    'patient', 'slide_name', 'xml_path', 'width', 'height',
    'total_regions', 'total_instances', 'labels_present', 'output_mask_dirs'
]

# Initialize master CSV if not present for saving summary info
if not os.path.exists(master_csv_path):
    with open(master_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(master_csv_header)

patients = [x[0] for x in os.walk(data_path)]

global_instance_counter = 0  # Optional global counter across all slides

for patient_loc in patients:
    # Compute patient name relative to data_path
    if patient_loc == data_path:
        continue

    rel = os.path.relpath(patient_loc, data_path)
    patient_name = '.' if rel == '.' else rel.split(os.sep)[0]
    if patient_name in ('.', ''):
        continue

    print(f"\nPatient: {patient_name}")

    # Patient mask directory
    patient_mask_dir = os.path.join(masks_root, patient_name)
    os.makedirs(patient_mask_dir, exist_ok=True)

    # Read .svs images under this patient folder
    sub_images = glob(os.path.join(patient_loc, '*.svs'))
    for sub_image_loc in sub_images:
        gt = 0  # instance counter within this single slide
        base_name = os.path.basename(sub_image_loc)
        sub_image_name = os.path.splitext(base_name)[0]
        print(f"  Slide: {sub_image_name}")

        # Per-slide output root (masks)
        sub_image_dir = os.path.join(patient_mask_dir, sub_image_name)
        os.makedirs(sub_image_dir, exist_ok=True)

        # Open slide
        try:
            slide = openslide.OpenSlide(sub_image_loc)
        except Exception as e:
            print(f"    [ERROR] Could not open slide: {sub_image_loc} | {e}")
            continue

        width, height = slide.level_dimensions[0]
        # Prepare per-label mask when label switches
        current_label = None
        n_ary_mask = None  # created per-label

        # Optional: Save slide as TIFF
        if save_tif_slide:
            try:
                rgba = np.array(slide.read_region((0, 0), 0, (width, height)))
                # If alpha present, drop it for TIFF
                if rgba.shape[-1] == 4:
                    rgb = rgba[:, :, :3]
                else:
                    rgb = rgba
                tiff_out = os.path.join(sub_image_dir, f"{sub_image_name}.tif")
                cv2.imwrite(tiff_out, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
            except Exception as e:
                print(f"    [WARN] Could not export TIFF for slide {sub_image_name}: {e}")

        # XML path
        xml_file_name = os.path.splitext(sub_image_loc)[0] + '.xml'
        if not os.path.exists(xml_file_name):
            print(f"    [WARN] Missing XML for slide: {xml_file_name}")
            continue

        # Parse XML File
        try:
            tree = ET.parse(xml_file_name)
            root = tree.getroot()
        except Exception as e:
            print(f"    [ERROR] Failed parsing XML {xml_file_name}: {e}")
            continue

        # ---------- METADATA COLLECTION ----------
        # Structure:
        # {
        #   'patient': ...,
        #   'slide_name': ...,
        #   'xml_path': ...,
        #   'dimensions': {'width': int, 'height': int},
        #   'labels': {
        #       'Epithelial': {
        #           'regions_count': int,
        #           'instance_indices': [1,2,...],
        #           'areas': [float,...]  # in px^2
        #       },
        #       ...
        #   },
        #   'total_regions': int,
        #   'total_instances': int,
        #   'generated_mask_dirs': ['.../Epithelial', '...']
        #   'generated_at': timestamp
        # }
        meta = {
            'patient': patient_name,
            'slide_name': sub_image_name,
            'xml_path': xml_file_name,
            'dimensions': {'width': width, 'height': height},
            'labels': {},
            'total_regions': 0,
            'total_instances': 0,
            'generated_mask_dirs': [],
            'generated_at': datetime.utcnow().isoformat() + 'Z'
        }

        # MASK + REGION LOOP
        # The provided XML structure seems like: root[k] groups; each group has 'Attribute' (label) & 'Region' entries
        # We create a new mask per label and save all its instances into that mask file (incrementing instance IDs).
        # When label changes, we flush the previous label mask to disk.

        def flush_label_mask(label_name, mask_arr, out_dir, instance_count):
            """Save mask for a completed label"""
            if label_name is None or mask_arr is None:
                return None
            os.makedirs(out_dir, exist_ok=True)
            # Build mask filename
            mask_path = os.path.join(out_dir, f"{sub_image_name}_{label_name}_mask.tif")
            # Save as 16-bit TIFF
            cv2.imwrite(mask_path, mask_arr.astype(mask_dtype))
            return mask_path

        # We’ll store output dirs (one per label)
        label_output_dirs = set()

        for k in range(len(root)):
            # Try to read a label from the first attribute block if present
            try:
                initial_label_candidates = [x.attrib.get('Name', '') for x in root[k][0]]
                if initial_label_candidates:
                    current_label = initial_label_candidates[0]
                else:
                    current_label = None
            except Exception:
                current_label = None

            # Ensure metadata bin for this label
            if current_label and current_label not in meta['labels']:
                meta['labels'][current_label] = {
                    'regions_count': 0,
                    'instance_indices': [],
                    'areas': []
                }

            # Prepare mask for current label (if any)
            if current_label is not None and n_ary_mask is None:
                n_ary_mask = np.zeros((height, width), dtype=mask_dtype)

            # Iterate children: look for Attribute changes, and Region geometries
            for child in root[k]:
                for x in child:
                    tag_name = x.tag

                    if tag_name == 'Attribute':
                        # Flush previous label mask if switching labels
                        new_label = x.attrib.get('Name', None)
                        if new_label != current_label:
                            # Flush old label
                            if current_label is not None and n_ary_mask is not None:
                                out_dir = os.path.join(sub_image_dir, current_label)
                                saved_path = flush_label_mask(current_label, n_ary_mask, out_dir, gt)
                                if saved_path:
                                    label_output_dirs.add(out_dir)
                            # Reset for new label
                            current_label = new_label
                            if current_label and current_label not in meta['labels']:
                                meta['labels'][current_label] = {
                                    'regions_count': 0,
                                    'instance_indices': [],
                                    'areas': []
                                }
                            n_ary_mask = np.zeros((height, width), dtype=mask_dtype)

                    elif tag_name == 'Region':
                        # Extract polygon vertices
                        try:
                            vertices = x[1]  # Typically 'Vertices'
                        except Exception:
                            continue

                        coords = np.zeros((len(vertices), 2), dtype=float)
                        for i, vertex in enumerate(vertices):
                            coords[i, 0] = float(vertex.attrib['X'])
                            coords[i, 1] = float(vertex.attrib['Y'])

                        # Update counts
                        meta['total_regions'] += 1
                        gt += 1
                        global_instance_counter += 1

                        # Rasterize polygon
                        # Note: draw.polygon expects row (y) and col (x) arrays in (H, W) image
                        xs = coords[:, 0]
                        ys = coords[:, 1]
                        rr, cc = draw.polygon(ys, xs, n_ary_mask.shape)

                        # Write instance id into mask
                        n_ary_mask[rr, cc] = gt

                        # Update per-label metadata (if label present)
                        if current_label:
                            meta['labels'][current_label]['regions_count'] += 1
                            meta['labels'][current_label]['instance_indices'].append(int(gt))
                            try:
                                area = Polygon(coords).area  # area in pixel units
                            except Exception:
                                area = float(len(rr))  # fallback: pixel count covered
                            meta['labels'][current_label]['areas'].append(float(area))

        # Flush final label mask
        if current_label is not None and n_ary_mask is not None:
            out_dir = os.path.join(sub_image_dir, current_label)
            saved_path = flush_label_mask(current_label, n_ary_mask, out_dir, gt)
            if saved_path:
                label_output_dirs.add(out_dir)

        # Finalize metadata
        meta['total_instances'] = int(gt)
        meta['generated_mask_dirs'] = sorted(label_output_dirs)
        labels_present = sorted(list(meta['labels'].keys()))

        # ---------- WRITE PER-SLIDE METADATA DOCS ----------
        # JSON (machine-readable)
        json_path = os.path.join(meta_root, f"{patient_name}__{sub_image_name}.json")
        with open(json_path, 'w') as jf:
            json.dump(meta, jf, indent=2)

        # Markdown (human-readable "doc")
        md_path = os.path.join(meta_root, f"{patient_name}__{sub_image_name}.md")
        with open(md_path, 'w') as mf:
            mf.write(f"# Slide Metadata: {sub_image_name}\n\n")
            mf.write(f"- **Patient**: `{patient_name}`\n")
            mf.write(f"- **XML Path**: `{xml_file_name}`\n")
            mf.write(f"- **Dimensions**: {width} × {height} px\n")
            mf.write(f"- **Generated At (UTC)**: {meta['generated_at']}\n")
            mf.write(f"- **Total Regions**: {meta['total_regions']}\n")
            mf.write(f"- **Total Instances (IDs)**: {meta['total_instances']}\n")
            mf.write(f"- **Mask Output Dirs**:\n")
            for d in meta['generated_mask_dirs']:
                mf.write(f"  - `{d}`\n")

            mf.write("\n## Labels Summary\n")
            if not labels_present:
                mf.write("_No labels detected in XML._\n")
            else:
                for lbl in labels_present:
                    entry = meta['labels'][lbl]
                    rc = entry['regions_count']
                    mf.write(f"\n### {lbl}\n")
                    mf.write(f"- Regions: {rc}\n")
                    if rc > 0:
                        ids_str = ', '.join(map(str, entry['instance_indices'][:20]))
                        if len(entry['instance_indices']) > 20:
                            ids_str += ', ...'
                        mf.write(f"- Instance IDs (sample): {ids_str}\n")

                        # Some quick area stats if available
                        areas = entry['areas']
                        if areas:
                            a_min = min(areas)
                            a_max = max(areas)
                            a_mean = sum(areas) / len(areas)
                            mf.write(f"- Area (px²): min={a_min:.1f}, mean={a_mean:.1f}, max={a_max:.1f}\n")

        # ---------- APPEND TO MASTER CSV ----------
        with open(master_csv_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                patient_name,
                sub_image_name,
                xml_file_name,
                width,
                height,
                meta['total_regions'],
                meta['total_instances'],
                ';'.join(labels_present),
                ';'.join(meta['generated_mask_dirs'])
            ])

        print(f"       Metadata saved:\n"
              f"       - JSON: {json_path}\n"
              f"       - DOC : {md_path}\n"
              f"       - Updated master CSV")

print("\nAll done.")



Patient: TrainingImages_and_Annotations

Patient: TrainingImages_and_Annotations
  Slide: TCGA-55-1594-01Z-00-DX1_001
    ✅ Metadata saved:
       - JSON: Output_Folder\metadata\TrainingImages_and_Annotations__TCGA-55-1594-01Z-00-DX1_001.json
       - DOC : Output_Folder\metadata\TrainingImages_and_Annotations__TCGA-55-1594-01Z-00-DX1_001.md
       - Updated master CSV
  Slide: TCGA-55-1594-01Z-00-DX1_002
    ✅ Metadata saved:
       - JSON: Output_Folder\metadata\TrainingImages_and_Annotations__TCGA-55-1594-01Z-00-DX1_002.json
       - DOC : Output_Folder\metadata\TrainingImages_and_Annotations__TCGA-55-1594-01Z-00-DX1_002.md
       - Updated master CSV
  Slide: TCGA-55-1594-01Z-00-DX1_003
    ✅ Metadata saved:
       - JSON: Output_Folder\metadata\TrainingImages_and_Annotations__TCGA-55-1594-01Z-00-DX1_003.json
       - DOC : Output_Folder\metadata\TrainingImages_and_Annotations__TCGA-55-1594-01Z-00-DX1_003.md
       - Updated master CSV
  Slide: TCGA-55-1594-01Z-00-DX1_004
    ✅ Me